In [ ]:
import numpy as np
import pandas as pd
import yaml
from pandarallel import pandarallel
from tqdm import tqdm

pandarallel.initialize(progress_bar=True, nb_workers=16)

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## Import data

In [ ]:
wos_classification = pd.read_parquet(dataset_config['path_processed'] + 'WOS/POST01_WOS_paper_CN.parquet')
wos_classification

## Add Entity List info

In [ ]:
def locate_entity(entry):

    entities_2018 = [
        "China Electronics Technology Group Corporation 13th Research Institute​", "China Electronics Technology Group Corporation 38th Research Institute​", "China Electronics Technology Group Corporation 55th Research Institute", 
        "China Volant Industry", "Hebei Far East Communication System Engineering", "Chengdu Jiashi", "Chengdu HiWafer Semiconductor", 
        "Fujian Jinhua Integrated Circuit"
    ]

    entities_2019 = [
        "Chengdu Haiguang", "Huawei", "Dawning Information", "Wuxi Jiangnan Institute of Computing Technology​", 
        "China General Nuclear Power", "China Nuclear Power Technology", "Suzhou Nuclear Power", "Zhejiang Dahua", "Hikvision", "iFLYTEK", 
        "Megvii", "SenseTime", "Xiamen Meiya Pico", "Shanghai Yitu", "Yixin Science and Tech"
    ]

    entities_2020 = [
        "Wuhan Yiersen", "Beijing Zhongyun Rongxin", "Beijing Jincheng Hanyu", "China Jiuyuan", "Harbin Chuangyue", "Harbin Yunlida", "Qihoo 360", "Shanghai Nova Instruments", 
        "Sichuan Dingcheng", "Sichuan Xintianyuan", "Sichuan Tuske", "Lijian Tianyan", "Kuaijisong", "Aksu Huafu Color Spinning", "CloudWalk", "FiberHome", 
        "Nanjing Fenghuo StarrySky", "NetPosa Technologies", "SenseNets", "Intellifusion Technologies", "Easyway", "BGI Genomics", "Beijing Genomics Institute", "Changji Esquel Textile", "Hefei Baolongda", "Hefei Meiling", 
        "Hetian Haolin Hair", "Hetian Teda Garments", "Jinchuang", "Nanjing Xinyimian", "Nanchang O-Film", "Tanyuan Technology", "CCCC Dredging", 
        "CCCC Tianjin", "CCCC Shanghai", "CCCC Guangzhou", "CCCC Second Harbor Engineering", "Beijing Huanjia", "Changzhou Guoguang", "China Electronics Technology Group Corporation No. 7 Research Institute", 
        "Beijing Chongxin Bada", "Guangzhou Guangyou", "Guangzhou Haige", "Guilin Changhai", "Hubei Guangxing", "Shaanxi Changling Electronic", "Shanghai Kaibo", 
        "Beijing Telixin Electronic", "Tianjin Broadcasting Equipment", "Tianjin 764", "Wuhan Mailet", "Wuxi Zhongde Meilian Biotechnology", "China Communications Construction Company", "CCCC", "China National Scientific Instruments and Materials", 
        "Chongqing Chuandong Shipbuilding", "CSSC Huangpu Wenchong", "DJI Technology", "Guangxin Maritime Heavy", "Guangzhou Taicheng Shipbuilding", "Jiangsu Hengxiang", "Shenzhen Guangui", 
        "Nanjing Asset Management", "Ningbo Semiconductor", "North Microelectronics", "Semiconductor Manufacturing International", "SMIC", "SMIC South Integrated Circuit", "SMIC Changdian Semiconductor", "SMIC Holdings", "SMIC Northern Integrated Circuit", 
        "SMIC Semiconductor", "Nuctech Company", "Tsinghua Tongfang", "Aero Engine Corporation of China", "AECC", "Aero Engine Corporation of China", "AECC Power", "AECC Gas Turbine", "AECC Commercial Aircraft Engine", 
        "AECC Harbin Dongan Engine", "AECC Shenyang", "AECC Hunan", "Anhui Yingliu Aerospace Power", "Aviation Industry Corporation of China", "AVIC", "AVIC Aircraft", 
        "Chengdu Aircraft Industry", "AVIC General", "Zhejiang Institute of AVIC", "AVIC International", "Beijing Baimu", "Beijing Andatek",
        "Beijing Liweier Aviation", "Guangming Yuanda Electronics", "Xi’an Aerospace Propulsion", "Chengdu Hangli Aviation", "AVIC Aviation Standard Parts",
        "CSSC Xigong", "Shanghai Yiling Electronic", "Hefei Fuhua Precision Machinery", "Guangzhou Hangxin", "Guizhou Hanyu Aviation",
        "Guizhou Liyang International Manufacturing", "AVIC Helicopter", "Hangzhou Bearing Test", "Harbin General Aircraft", "Henan Aerospace Precision",
        "Hunan Southern General", "Chongqing AopuTai Communication", "Jiangsu Meilong", "Hubei Hanyu Jiatai", "Jincheng Group",
        "Medix Laboratory", "Shaanxi Aero-Electrics", "Shaanxi Aircraft Industry", "Shanghai Aerospace Equipment", "Shanghai Aircraft Manufacturing",
        "Shanghai Tianlang Electronic", "Shenyang Instrument Science Research", "Shenyang Aircraft", "Neiyang Xizi Aviation", "Sichuan Hangte Aviation",
        "Chengtai Aviation", "Jiangsu SUMEC Instruments", "Suzhou Yike", "Wuxi Hanger", "Wuxi Parker New Material",
        "Wuxi Turbine Blade", "Xi’an Aircraft Industry Avionics", "Xifei Science & Technology", "AECC Xi’an Aero-Engine Controls", "Xi’an Aircraft Industry",
        "Xi’an Xihang Group", "Xi’an Xiluo Aviation", "Yibin Sanjiang Machinery", "Zhejiang Wanmei"
    ]

    entities_2021 = [
        "China National Offshore Oil", "CNOOC", "Beijing Tianjiao Aviation", "Xinwei Microelectronics", "Tianjin Phytium", "Hoshine Silicon", "Xinjiang Daqo New Energy",
        "Xinjiang Eastern Hope", "Xinjiang GCL New Energy", "Beijing Eastsoft Junyue", "Beijing Yanjing Electronics", "Beijing Geling Shentong",
        "Beijing Haili United", "Beijing Zhongdian Xingfa", "Chengdu Xiwu Xin’an Intelligent System", "West InfoSec Intelligent System", "Hangzhou Hualan Microelectronics", "Shanghai Jinzhuo",
        "Beijing Eastsoft", "Li’ang Tech", "Shenzhen Kepa Information", "Shenzhen Huaantai", "Suzhou Keda",
        "Tongfang Ruian", "Urumqi Tianyao Weiye", "Wuhan Raycus Fiber Laser", "Xin Zui Beidou Tongchuang", "Xinjiang Lianhai Chuangzhi",
        "Xinjiang Xiling Information", "Xinjiang Tangli", "Shenzhen Jia’zhao", "Hangzhou Zhongke Microelectronics", "Hunan Goke Microelectronics",
        "H3C Semiconductor", "Poly Asia-Pacific", "QuantumCTek", "Shaanxi Zhien Electromechanical", "Shanghai Guodun Quantum",
        "Xi’an Aerospace Huaxun", "Suzhou Yunxin Microelectronics", "Aerospace Zanguang", "Zhanguang", "Changsha Jingjia Microelectronics", "Kontel Tech",
        "Fujian Torch Electronic", "Hangzhou Hikvision", "Huahai Communication", "Hong Kong Changhua Electronics", "Huashijie Electronics", "Hua Vision Electronics", 
        "Super Systems Alliance", "Inner Mongolia First Machinery", "Jiangsu Hengtong Marine Optical", "Jiangsu Hengtong Optic-Electric", "Shaanxi Yacheng Microelectronics",
        "Shanghai Aisino Hangxin Electronic", "Shanghai Zuoshi Control", "Shenzhen Ruifen Technology", "Refon Technology", "Mingyang Electric", "Zhongtian Technology Submarine Cable", "ZTT Submarine Cable"
    ]

    entities_2022 = [
        "Keqin Ket Electronics", "Hong Kong Shijieta", "Xinnuo Electronics", "Signo Electronics", "Weike Electronics", "Vico Electronics", "Wangnianhua Electronics",
        "Beijing Hailanxin Data", "CSSC Electronics", "Sansha Hailanxin Data", "Sanya Hailan Huanyu Marine ", "Anhui Cambricon",
        "Anzhou International Group", "Beijing Huatian Haifeng", "Beijing Machinery Industry Automation", "Beijing UniStrong", "Beijing Yinjing",
        "Cambricon Technologies", "CETC Cloud", "China Electronics Lais", "Guangdong QinZhi", "Hefei Zhaoxin Electronics",
        "Nanjing Aixi Information", "Nanjing Lais Cyber", "Nanjing Lais Electronic", "Nanjing Lais Information", "Shenzhen Pengxin Micro Integrated",
        "Shanghai Cambricon", "Shanghai Micro Electronics", "Shanghai Suowei Information", "Suzhou Cambricon Information", "Liyang 28th Institute Systems Equipment",
        "Tianjin Tiandy Technologies", "Xiong’an Cambricon", "Yangtze Memory", "Beijing Zhongke Xinlian", "Yangtze Memory"
    ]

    entities_2023 = [
        "Beijing Nanjiang Aerospace", "Dongguan Lingkong Remote", "Yigesiman Aviation", "Guangzhou Tianhaixiang", "Shanxi Yigesiman",
        "AOOK Tech", "Beijing Tiantai", "Beijing Yunze", "De Aerospace", "Deyuhang Tech", "Changsha Wuyi Space",
        "Beijing Fourth Paradigm", "Quanxiang International Freight", "Aisu Industrial", "Aisu Industries", "Baoding Giant", "Baoding Shizitong Enterprise",
        "Beijing Zhengyuan Chuangshi", "BGI Forensic", "Baoding Kaituo Precision", "Inspur Group", "Loongson Tech",
        "Nanjing Gelunba", "Nanjing Jiuding", "Shanghai Shunmiao", "Suzhou Shengke Communication", "Suzhou Shengke Tech",
        "Luopu Haishi Dingxin Electronic", "Moyu Haishi Electronic", "Pishan Haishi Yongan", "Urumqi Haishi Xin’an Electronic", "Yutian Haishi Meitian Electronic",
        "Liwei Tech", "Yishang Network", "Eshang Network", "Yongli Electronic", "AVIC International", "China Aviation Technology",
        "Beijing Yiweixun Tongchuang", "Beijing Luoluo Tech", "Beijing Ruiyuan Wende", "Beijing Tianshenghua Information", "Beijing Tianshenghua Tech",
        "Beierte Consulting", "Belltech Consulting", "Changzhou Youtaike", "Chengdu Boyang", "China Tianli Aviation", "Yiqiang International Trade",
        "Xianfeng Services", "Xanfeng Services", "Dazhong Technology", "Volks Technology", "Xinxin Industrial Investment", "Outang", "Anshi Asia-Pacific", "Ansys AP Tech", 
        "Shenzhen Qianpu", "Shanghai Aerospace", "Shanghai Qingfeng Tech", "Shanghai Qingfeng Technology", "Shanghai Chaosuan", "Shanghai Supercomputing Tech", 
        "Jinda Electronics", "General Enterprise", "Xinjiang Kehua Hechang", "Asia-Pacific Link", "AsiaPac Link", "Guilin Alpha Rubber",
        "Hangzhou Fuyang Ketuo", "Ruiwen International Trade", "Shenzhen Kasipu Tech", "Lianmeng Electronic Tech", "Alliance Electronics Tech", "Alpha Trade",
        "Shanghai Yalian", "Zhongyin Semiconductor", "Shenzhen Chaaisi Electronic", "Chengdu Jingxin Microwave", "China Shengshi International",
        "Yijing Tech", "Gelaite Electronic Tech", "Global Shell Customs Brokerage", "Huayuan Shitong", "Jingfu Circuit Boards",
        "Niuwotai International Trade", "Nuopuxun Electronic Tech", "Anshida Electronics", "PT Tech", "Rongbotong Semiconductor",
        "Shanghai Yingzhong", "Shenzhen Yishida", "Shiwabei Optoelectronics", "Sentuo Semiconductor", "Sento Semiconductor", "Quanzhou Nan’an Teyitong Electronics",
        "Youchuang PCB", "Utron PCB", "Yongqi International", "Zeyuan Tech", "Beijing Biren Tech", "Guangzhou Biren",
        "Hangzhou Biren", "Light Cloud", "Guangxian Cloud", "Moore Threads", "Shanghai Biren", "Chaoran Semiconductor", "Chaoran Semi", 
        "Shanghai Xinzhili Enterprise", "Zhuhai Biren", "Pride Tech"
    ]

    entities_2024 = [
        "Shenzhen Sipide Industrial", "Shenzhen Speed Industrial", "Shihe Tech", "United Electronics", "Chengdu Beizhan Electronics", "Beijing Anhuaixin", "Jiangxi Xintuo", "Lianzhong Cluster",
        "Shenzhen Jasbo", "Siteng Heli", "Xi’an Likechuangxin", "AEE Shenzhen Yidian", "Beijing BDStar Navigation",
        "Beijing Leike Defense", "Beijing Ruidakang", "Beijing Tianhaida", "Beijing Zhongshang Dingsheng", "CETC Chip Tech",
        "CETC SiyI Tech", "CETC SiYi Tech", "Chengdu Huaricom", "Chengdu Zongheng Automation", "China Electronics Technology Group Corporation Electronic Equipment Division", "CETC Electronic Equipment Division", "CSIC Pengli",
        "SuperMap", "Star Map", "Hexin Xingtong", "Origin Quantum Computing", "Origin Quantum", "Shenzhen Yidian", "Suzhou Telecommunication Motor",
        "Taiyuan Yifute Equipment", "United Microelectronics Center", "Xi’an Hengda Microwave", "Zhongke Star Map Space"
    ]

    entities_by_year = {
        2018: entities_2018,
        2019: entities_2019,
        2020: entities_2020,
        2021: entities_2021,
        2022: entities_2022,
        2023: entities_2023,
        2024: entities_2024
    }

    
    for yr in entities_by_year:
        for elem in entities_by_year[yr]:
            if elem.lower() in entry.lower():
                return yr
    return False


In [ ]:
locate_entity('huawei')

In [ ]:
# Exceution time: 7 min
wos_classification['Entity_list'] = wos_classification['affiliationame'].parallel_apply(locate_entity)
wos_classification

In [ ]:
true_count = (wos_classification['Entity_list'] != False).sum()
print("Number of TRUE values:", true_count)

In [ ]:
wos_classification[wos_classification['Entity_list'] != False]['affiliationame'].value_counts().head(60)

## Add variables

In [ ]:
wos_classification[wos_classification['type'] == 'f']

In [ ]:
wosid_with_firm = wos_classification[wos_classification['type'] == 'f']['wosid'].unique()
wosid_with_firm

In [ ]:
df_with_firm = wos_classification[wos_classification['wosid'].isin(wosid_with_firm)]
df_with_firm

In [ ]:
df_with_firm_dedup = df_with_firm.drop_duplicates(subset=['wosid', 'affiliationame'])[['wosid', 'affiliationame', 'type', 'Entity_list']]
df_with_firm_dedup

In [ ]:
# Add the fractional_num column
df_with_firm_dedup['fractional_num'] = 1 / df_with_firm_dedup.groupby('wosid')['wosid'].transform('count')
df_with_firm_dedup

In [ ]:
# Export papers published by firms for further analysis
df_with_firm_dedup[['wosid', 'affiliationame', 'fractional_num']].to_parquet(dataset_config['path_processed'] + 'WOS/WOS_paperid_CNfirm.parquet')

In [ ]:
for wosid, df in tqdm(df_with_firm_dedup.groupby(by='wosid')):
    if df['type'].apply(lambda x: 'u' in x).any():
        break

In [ ]:
results = []

for wosid, df in tqdm(df_with_firm_dedup.groupby(by='wosid')):
    df['with_uni'] = df['type'].apply(lambda x: 'u' in x).any()
    results.append(df)

In [ ]:
df_with_firm_uni = pd.concat(results)
df_with_firm_uni

In [ ]:
df_with_firm_uni['Entity_list'].value_counts()

### (1) Merge with paper info

In [ ]:
wos_paper_info = pd.read_parquet(dataset_config['path_processed'] + 'WOS/WOS_paper_level.parquet').drop_duplicates()
wos_paper_info.rename(columns={'pub year': 'year'}, inplace=True)
wos_paper_info

In [ ]:
df_with_firm_uni_paperinfo = pd.merge(df_with_firm_uni, wos_paper_info, on='wosid', how='left')
df_with_firm_uni_paperinfo

### (2) Merge with ISSN (delete the paper already in the CNKI)

In [ ]:
wos_cnki_overlap = pd.read_csv(dataset_config['path_processed'] + 'CNKI/overlap_cnkiwos_issn.csv', usecols=['journal_WOS'])
overlap_journals = wos_cnki_overlap['journal_WOS'] # Get the list of journals to remove
overlap_journals

In [ ]:
# Drop rows where the journal is in the overlap list
df_drop_overlap = df_with_firm_uni_paperinfo[
    ~df_with_firm_uni_paperinfo['journal'].isin(overlap_journals)
]
df_drop_overlap

### (3) Merge with JIF

In [ ]:
wos_jif = pd.read_parquet(dataset_config['path_processed'] + 'WOS/WOS_JIF.parquet')
wos_jif

In [ ]:
wos_all = pd.merge(df_drop_overlap, wos_jif, on='wosid', how='left')
wos_all

## Discriptive statistics: JIF

In [ ]:
# (1) Keep three columns
wos_filtered = wos_all[['wosid', 'year', 'impact_factor']].copy()

# (2) Drop duplicates
wos_filtered = wos_filtered.drop_duplicates()

# (3) Filter to 2010-2022
wos_period = wos_filtered[(wos_filtered['year'] >= 2000) & (wos_filtered['year'] <= 2022)]

# Overall mean
overall_mean = wos_period['impact_factor'].mean()
print("2010–2022年总体均值 JIF:", overall_mean)

# (4) Annual mean
yearly_means = wos_period.groupby('year')['impact_factor'].mean()
print("\n每年平均 JIF:")
print(yearly_means)

wos_period.to_csv(dataset_config['path_processed'] + 'WOS/WOS_CNfirm_JIF.csv', index=False)


In [ ]:
stats = (
    wos_all["impact_factor"]
      .agg(max_value   = "max",
           min_value   = "min",
           mean_value  = "mean",
           median_value= "median",
           p66_value    = lambda x: x.quantile(0.66))
      .round(1)
)

stats

## To firm-year level

In [ ]:
# 2 min
aff_stats = {
    "affiliationame": [],
    "year": [],
    "num_papers": [],
    "num_with_uni": [],
    "in_entity_list": [],
    "WOS_field": [],
    "num_papers_h": [],
    "num_with_uni_h": [],
    "num_papers_hh": [],
    "num_with_uni_hh": [],
    "avg_jif": [],
    "num_papers_fractional": []
}


for aff_name, df in (
        wos_all.query("type == 'f'")
        .groupby("affiliationame")
):
    
    most_common_wos_area = df["WOS area"].mode()[0] if not df["WOS area"].mode().empty else None

    for year, sub_df in df.groupby("year"):
        aff_stats["affiliationame"].append(aff_name)
        aff_stats["year"].append(year)
        aff_stats["num_papers"].append(len(sub_df))
        aff_stats["num_with_uni"].append(sub_df["with_uni"].sum())
        aff_stats["in_entity_list"].append(sub_df["Entity_list"].iloc[0])
        aff_stats["WOS_field"].append(most_common_wos_area)

        
        # NEW: Mean JIF, skipping missing values.
        jif_mean = sub_df["impact_factor"].dropna().mean()
        aff_stats["avg_jif"].append(jif_mean)

        hi = sub_df[sub_df["impact_factor"] > 3.1]
        aff_stats["num_papers_h"].append(len(hi))
        aff_stats["num_with_uni_h"].append(hi["with_uni"].sum())

        hihi = sub_df[sub_df["impact_factor"] > 4.4]
        aff_stats["num_papers_hh"].append(len(hihi))
        aff_stats["num_with_uni_hh"].append(hihi["with_uni"].sum())

        aff_stats["num_papers_fractional"].append(sub_df["fractional_num"].sum())

In [ ]:
df_aff_stats = pd.DataFrame(aff_stats)
df_aff_stats

In [ ]:
# generate firm ID and field ID
df_aff_stats['firm_id'] = pd.factorize(df_aff_stats['affiliationame'])[0] + 1
df_aff_stats['field_id'] = pd.factorize(df_aff_stats['WOS_field'])[0] + 1
df_aff_stats

In [ ]:
df_aff_stats.to_csv(dataset_config['path_processed'] + 'WOS/WOS_CNfirm_publication.csv', index=False)

## Statistics

In [ ]:
df_aff_stats[df_aff_stats['num_papers'] > 600]

In [ ]:
# Firms listed on the Entity List 
df_aff_stats[(df_aff_stats['in_entity_list']) & (df_aff_stats['num_papers'] > 100)]

In [ ]:
# Firms not listed on the Entity List 
df_aff_stats[(~df_aff_stats['in_entity_list']) & (df_aff_stats['num_papers'] > 1000)]